In [1]:
import numpy as np
from tqdm import tqdm
from scipy import ndimage
import os.path
import math

def dissmeasure_vectorized(fvec, amp):
    """Fully vectorized Plomp-Levelt dissonance model"""
    # Sort by frequency
    idx = np.argsort(fvec)
    fr_sorted = fvec[idx]
    am_sorted = amp[idx]

    Dstar = 0.24
    S1 = 0.0207
    S2 = 18.96
    C1 = 5.0
    C2 = -5.0
    A1 = -3.51
    A2 = -5.75

    n = len(fr_sorted)
    
    # Create pairwise matrices using broadcasting
    # Shape: (n, n)
    Fmin_matrix = fr_sorted[:, np.newaxis]  # (n, 1)
    Fmax_matrix = fr_sorted[np.newaxis, :]  # (1, n)
    
    # Only compute upper triangle (i < j)
    i_indices, j_indices = np.triu_indices(n, k=1)
    
    Fmin = fr_sorted[i_indices]
    Fdif = fr_sorted[j_indices] - Fmin
    
    S = Dstar / (S1 * Fmin + S2)
    a = np.minimum(am_sorted[i_indices], am_sorted[j_indices])
    SFdif = S * Fdif
    
    # Vectorized exponential computation
    total = np.sum(a * (C1 * np.exp(A1 * SFdif) + C2 * np.exp(A2 * SFdif)))
    
    return total


def calculate_dissonance_batch(freq_base, amp_base, alpha, beta, gamma_array):
    """Calculate dissonance for multiple gamma values at once
    
    Args:
        freq_base: Base frequencies (num_harmonics,)
        amp_base: Base amplitudes (num_harmonics,)
        alpha: Single alpha value
        beta: Single beta value
        gamma_array: Array of gamma values (n_gamma,)
    
    Returns:
        Array of dissonance values (n_gamma,)
    """
    num_harmonics = len(freq_base)
    n_gamma = len(gamma_array)
    
    # Create frequency array for all gamma values at once
    # Shape: (n_gamma, 4 * num_harmonics)
    all_freq = np.zeros((n_gamma, 4 * num_harmonics), dtype=np.float32)
    
    # Base frequencies (same for all)
    all_freq[:, :num_harmonics] = freq_base
    all_freq[:, num_harmonics:2*num_harmonics] = freq_base * alpha
    all_freq[:, 2*num_harmonics:3*num_harmonics] = freq_base * beta
    
    # Gamma frequencies (different for each)
    all_freq[:, 3*num_harmonics:] = freq_base * gamma_array[:, np.newaxis]
    
    # Create amplitude array (same for all)
    all_amp = np.tile(amp_base, 4)
    
    # Calculate dissonance for each gamma
    dissonances = np.array([dissmeasure_vectorized(all_freq[i], all_amp) 
                           for i in range(n_gamma)])
    
    return dissonances


def find_harmonic_nodes_vectorized(
    alpha_range, beta_range, gamma_range, dissonance_3d, num_nodes=77, filter_size=5
):
    """Find local minima using scipy.ndimage for faster computation"""
    print(f"\nFinding {num_nodes} harmonic nodes...")

    # Use scipy's maximum_filter to find local minima
    # A point is a local minimum if it's smaller than all neighbors
    footprint = np.ones((filter_size*2+1, filter_size*2+1, filter_size*2+1))
    local_min = ndimage.minimum_filter(dissonance_3d, footprint=footprint)
    
    # Points where value equals local minimum are local minima
    is_local_min = (dissonance_3d == local_min) & (~np.isnan(dissonance_3d))
    
    # Get coordinates of local minima
    min_coords = np.argwhere(is_local_min)
    
    nodes = []
    step_size = (alpha_range[-1] - alpha_range[0]) / len(alpha_range)
    boundary_margin = max(3, int(0.1 * len(alpha_range)))
    prominence_radius = max(6, filter_size * 2)
    
    n_alpha, n_beta, n_gamma = dissonance_3d.shape
    
    for i, j, k in min_coords:
        # Skip boundaries
        if (i < boundary_margin or i >= n_alpha - boundary_margin or
            j < boundary_margin or j >= n_beta - boundary_margin or
            k < boundary_margin or k >= n_gamma - boundary_margin):
            continue
        
        alpha_val = alpha_range[i]
        beta_val = beta_range[j]
        gamma_val = gamma_range[k]
        value = dissonance_3d[i, j, k]
        
        # Check spacing
        if (abs(alpha_val - beta_val) < step_size * 2 or 
            abs(beta_val - gamma_val) < step_size * 2):
            continue
        
        # Calculate prominence using slicing (vectorized)
        i_start = max(0, i - prominence_radius)
        i_end = min(n_alpha, i + prominence_radius + 1)
        j_start = max(0, j - prominence_radius)
        j_end = min(n_beta, j + prominence_radius + 1)
        k_start = max(0, k - prominence_radius)
        k_end = min(n_gamma, k + prominence_radius + 1)
        
        region = dissonance_3d[i_start:i_end, j_start:j_end, k_start:k_end]
        max_in_radius = np.nanmax(region)
        prominence = max_in_radius - value
        
        if prominence < 0.001:
            continue
        
        # Calculate gradient using slicing (vectorized)
        i_start = max(0, i - 1)
        i_end = min(n_alpha, i + 2)
        j_start = max(0, j - 1)
        j_end = min(n_beta, j + 2)
        k_start = max(0, k - 1)
        k_end = min(n_gamma, k + 2)
        
        neighbors = dissonance_3d[i_start:i_end, j_start:j_end, k_start:k_end]
        gradient_sum = np.nansum(np.abs(neighbors - value)) - 0  # Subtract center
        gradient_count = np.sum(~np.isnan(neighbors)) - 1
        
        avg_gradient = gradient_sum / gradient_count if gradient_count > 0 else 0
        curvature = prominence * (1 + avg_gradient * 10)
        
        nodes.append({
            'alpha': alpha_val,
            'beta': beta_val,
            'gamma': gamma_val,
            'dissonance': value,
            'curvature': curvature
        })
    
    # Sort by curvature and take top N
    nodes.sort(key=lambda x: x['curvature'], reverse=True)
    return nodes[:num_nodes]


def generate_dataset_optimized(base_freq=220.0, nodes=400, harmonics=6, num_local_minima=77):
    """Generate 3D dissonance dataset with aggressive vectorization"""
    print(f"Generating dataset: {nodes}x{nodes}x{nodes} grid")
    print(f"Base frequency: {base_freq} Hz")
    print(f"Harmonics: {harmonics}")

    # Pre-compute base frequencies and amplitudes
    freq_base = base_freq * np.arange(1, harmonics + 1, dtype=np.float32)
    amp_base = np.ones(harmonics, dtype=np.float32)

    # Create ranges
    alpha_range = np.linspace(1.0, 2.0, nodes, dtype=np.float32)
    beta_range = np.linspace(1.0, 2.0, nodes, dtype=np.float32)
    gamma_range = np.linspace(1.0, 2.0, nodes, dtype=np.float32)

    # Initialize 3D array
    dissonance_3d = np.full((nodes, nodes, nodes), np.nan, dtype=np.float32)

    # Count valid computations for progress bar
    total_valid = 0
    for i in range(nodes):
        for j in range(nodes):
            if alpha_range[i] <= beta_range[j]:
                # Count valid k values where beta <= gamma
                total_valid += np.sum(beta_range[j] <= gamma_range)
    
    with tqdm(total=total_valid, desc="Computing dissonance") as pbar:
        for i in range(nodes):
            alpha = alpha_range[i]
            
            for j in range(nodes):
                beta = beta_range[j]
                
                # Skip if alpha > beta
                if alpha > beta:
                    continue
                
                # Find valid gamma indices (where beta <= gamma)
                valid_gamma_mask = gamma_range >= beta
                valid_gamma_indices = np.where(valid_gamma_mask)[0]
                
                if len(valid_gamma_indices) == 0:
                    continue
                
                # Get valid gamma values
                valid_gammas = gamma_range[valid_gamma_indices]
                
                # Batch compute dissonance for all valid gammas
                dissonances = calculate_dissonance_batch(
                    freq_base, amp_base, alpha, beta, valid_gammas
                )
                
                # Store results
                dissonance_3d[i, j, valid_gamma_indices] = dissonances
                
                pbar.update(len(valid_gammas))

    # Find local minima nodes
    raw_nodes = find_harmonic_nodes_vectorized(
        alpha_range, beta_range, gamma_range, dissonance_3d,
        num_nodes=num_local_minima, filter_size=5
    )

    # Refine nodes with stochastic search
    print(f"\nRefining {len(raw_nodes)} nodes with stochastic search...")
    refined_nodes = []
    for node in tqdm(raw_nodes, desc="Refining nodes"):
        refined = refine_node_stochastic(node, base_freq, harmonics, iterations=100)
        refined_nodes.append(refined)

    return alpha_range, beta_range, gamma_range, dissonance_3d, refined_nodes


def refine_node_stochastic(node, base_freq, num_harmonics, iterations=100):
    """Stochastic refinement - shake nodes to find true minimum"""
    best_alpha = node['alpha']
    best_beta = node['beta']
    best_gamma = node['gamma']
    best_diss = node['dissonance']

    initial_step = 0.015
    step_size = initial_step
    no_improvement = 0

    # Pre-allocate arrays for dissonance calculation
    freq_base = base_freq * np.arange(1, num_harmonics + 1, dtype=np.float32)
    amp_base = np.ones(num_harmonics, dtype=np.float32)
    all_freq = np.zeros(4 * num_harmonics, dtype=np.float32)
    all_amp = np.tile(amp_base, 4)
    all_freq[:num_harmonics] = freq_base

    for i in range(iterations):
        test_alpha = best_alpha + (np.random.random() - 0.5) * step_size
        test_beta = best_beta + (np.random.random() - 0.5) * step_size
        test_gamma = best_gamma + (np.random.random() - 0.5) * step_size

        if (test_alpha >= 1.0 and test_alpha <= 2.0 and
            test_beta >= 1.0 and test_beta <= 2.0 and
            test_gamma >= 1.0 and test_gamma <= 2.0 and
            test_alpha <= test_beta and test_beta <= test_gamma):

            # Update frequencies
            all_freq[num_harmonics:2*num_harmonics] = freq_base * test_alpha
            all_freq[2*num_harmonics:3*num_harmonics] = freq_base * test_beta
            all_freq[3*num_harmonics:] = freq_base * test_gamma

            test_diss = dissmeasure_vectorized(all_freq, all_amp)

            if test_diss < best_diss:
                best_alpha = test_alpha
                best_beta = test_beta
                best_gamma = test_gamma
                best_diss = test_diss
                no_improvement = 0
                step_size = initial_step
            else:
                no_improvement += 1

        if no_improvement > 10:
            step_size *= 0.8
            no_improvement = 0

    return {
        'alpha': best_alpha,
        'beta': best_beta,
        'gamma': best_gamma,
        'dissonance': best_diss
    }


def save_binary_dataset(
    alpha_range, beta_range, gamma_range, dissonance_3d, nodes, base_freq
):
    """Save dataset as binary file (Float32 format matching JS) in chunks of ~25MB"""
    # Use the existing dataset directory in the project root
    path = "dataset"

    # Prepare base filename
    base_filename = f"harmonic-{int(base_freq)}Hz-{len(alpha_range)}nodes"

    # Prepare nodes data
    nodes_flat = []
    for node in nodes:
        nodes_flat.extend(
            [node["alpha"], node["beta"], node["gamma"], node["dissonance"]]
        )
    nodes_data = np.array(nodes_flat, dtype=np.float32)

    # Calculate chunk size (in number of elements) for ~25MB chunks
    # Each float32 is 4 bytes
    target_chunk_size_bytes = 25 * 1024 * 1024  # 25MB
    chunk_size_elements = target_chunk_size_bytes // 4

    # Save metadata file (ranges and nodes)
    metadata = np.concatenate(
        [
            np.array([len(nodes)], dtype=np.float32),
            nodes_data,
            alpha_range.astype(np.float32),
            beta_range.astype(np.float32),
            gamma_range.astype(np.float32),
        ]
    )
    metadata_filename = os.path.join(path, f"{base_filename}-metadata.bin")
    with open(metadata_filename, "wb") as f:
        f.write(metadata.tobytes())

    # Flatten and chunk the 3D dissonance array
    flat_diss = dissonance_3d.flatten()
    num_chunks = math.ceil(len(flat_diss) / chunk_size_elements)

    chunk_filenames = []
    total_size_mb = 0

    for i in range(num_chunks):
        start_idx = i * chunk_size_elements
        end_idx = min((i + 1) * chunk_size_elements, len(flat_diss))
        chunk_data = flat_diss[start_idx:end_idx]

        chunk_filename = os.path.join(path, f"{base_filename}-chunk{i+1:03d}.bin")
        with open(chunk_filename, "wb") as f:
            f.write(chunk_data.tobytes())

        chunk_size_mb = len(chunk_data.tobytes()) / (1024 * 1024)
        total_size_mb += chunk_size_mb
        chunk_filenames.append(chunk_filename)

        print(f"Saved chunk {i+1:03d}: {chunk_filename}")
        print(f"Chunk size: {chunk_size_mb:.2f} MB")

    metadata_size_mb = len(metadata.tobytes()) / (1024 * 1024)
    total_size_mb += metadata_size_mb

    print(f"\nSaved metadata: {metadata_filename}")
    print(f"Metadata size: {metadata_size_mb:.2f} MB")
    print(f"Total size across all files: {total_size_mb:.2f} MB")
    print(f"Total chunks: {num_chunks}")
    print(f"Grid points: {len(flat_diss):,}")
    print(f"Local minima nodes: {len(nodes)}")

    return metadata_filename, chunk_filenames


if __name__ == "__main__":
    # Configuration
    BASE_FREQ = 220.0
    NODES = 400
    HARMONICS = 6
    LOCAL_MINIMA = 77

    print(f"Estimated time: ~{NODES**3 / 10000 / 60:.1f} minutes")
    print(f"Total points: {NODES**3:,}\n")

    # Generate dataset
    alpha_range, beta_range, gamma_range, dissonance_3d, refined_nodes = (
        generate_dataset_optimized(
            base_freq=BASE_FREQ,
            nodes=NODES,
            harmonics=HARMONICS,
            num_local_minima=LOCAL_MINIMA,
        )
    )

    # Save binary files in chunks
    metadata_file, chunk_files = save_binary_dataset(
        alpha_range, beta_range, gamma_range, dissonance_3d, refined_nodes, BASE_FREQ
    )

    print("\nDataset generation complete!")
    print(f"Use these files in your web app to skip computation.")

Estimated time: ~106.7 minutes
Total points: 64,000,000

Generating dataset: 400x400x400 grid
Base frequency: 220.0 Hz
Harmonics: 6


Computing dissonance: 100%|██████████| 10746800/10746800 [04:50<00:00, 36974.80it/s]



Finding 77 harmonic nodes...

Refining 77 nodes with stochastic search...


Refining nodes: 100%|██████████| 77/77 [00:00<00:00, 280.09it/s]


Saved chunk 001: dataset/harmonic-220Hz-400nodes-chunk001.bin
Chunk size: 25.00 MB
Saved chunk 002: dataset/harmonic-220Hz-400nodes-chunk002.bin
Chunk size: 25.00 MB
Saved chunk 003: dataset/harmonic-220Hz-400nodes-chunk003.bin
Chunk size: 25.00 MB
Saved chunk 004: dataset/harmonic-220Hz-400nodes-chunk004.bin
Chunk size: 25.00 MB
Saved chunk 005: dataset/harmonic-220Hz-400nodes-chunk005.bin
Chunk size: 25.00 MB
Saved chunk 006: dataset/harmonic-220Hz-400nodes-chunk006.bin
Chunk size: 25.00 MB
Saved chunk 007: dataset/harmonic-220Hz-400nodes-chunk007.bin
Chunk size: 25.00 MB
Saved chunk 008: dataset/harmonic-220Hz-400nodes-chunk008.bin
Chunk size: 25.00 MB
Saved chunk 009: dataset/harmonic-220Hz-400nodes-chunk009.bin
Chunk size: 25.00 MB
Saved chunk 010: dataset/harmonic-220Hz-400nodes-chunk010.bin
Chunk size: 19.14 MB

Saved metadata: dataset/harmonic-220Hz-400nodes-metadata.bin
Metadata size: 0.01 MB
Total size across all files: 244.15 MB
Total chunks: 10
Grid points: 64,000,000
Local